# Data Analysis

Families is human assesed so it could be good to utilize if zero shot training doesn't work out.

In [1]:
import pandas as pd
import numpy as np

In [2]:
tags_df = pd.read_csv("tags/families.csv", header=None, names = ["font", "tag", "value"], usecols=[0,2,3])

In [3]:
tags_df

,font,tag,value
0,42dot Sans,/Expressive/Business,75.0
1,42dot Sans,/Expressive/Loud,40.0
2,42dot Sans,/Quality/Concept,70.0
3,42dot Sans,/Quality/Drawing,70.0
4,42dot Sans,/Quality/Spacing,70.0
...,...,...,...
19535,Znamenny Musical Notation,/Quality/Concept,70.0
19536,Znamenny Musical Notation,/Quality/Drawing,70.0
19537,Znamenny Musical Notation,/Quality/Spacing,70.0
19538,Znamenny Musical Notation,/Quality/Wordspace,100.0


In [4]:
split_data = tags_df['tag'].str.split('/', expand=True).iloc[:, [1,2]]

In [5]:
split_data

,1,2
0,Expressive,Business
1,Expressive,Loud
2,Quality,Concept
3,Quality,Drawing
4,Quality,Spacing
...,...,...
19535,Quality,Concept
19536,Quality,Drawing
19537,Quality,Spacing
19538,Quality,Wordspace


In [6]:
tags_df[['tag_category', 'tag_value']] = split_data

Before we filter this down let's look at the distributions for different tags.

In [7]:
tags_df

,font,tag,value,tag_category,tag_value
0,42dot Sans,/Expressive/Business,75.0,Expressive,Business
1,42dot Sans,/Expressive/Loud,40.0,Expressive,Loud
2,42dot Sans,/Quality/Concept,70.0,Quality,Concept
3,42dot Sans,/Quality/Drawing,70.0,Quality,Drawing
4,42dot Sans,/Quality/Spacing,70.0,Quality,Spacing
...,...,...,...,...,...
19535,Znamenny Musical Notation,/Quality/Concept,70.0,Quality,Concept
19536,Znamenny Musical Notation,/Quality/Drawing,70.0,Quality,Drawing
19537,Znamenny Musical Notation,/Quality/Spacing,70.0,Quality,Spacing
19538,Znamenny Musical Notation,/Quality/Wordspace,100.0,Quality,Wordspace


In [8]:
tags_df["tag_category"].value_counts()

tag_category
Expressive     8458
Quality        7644
Sans           1095
Script          662
Theme           593
Seasonal        421
Serif           370
Purpose         132
Slab             88
Monospace        53
Special use      24
Name: count, dtype: int64

In [9]:
tags_df[tags_df["tag_category"] == "Quality"]["tag_value"].value_counts()

tag_value
Concept      1911
Drawing      1911
Spacing      1911
Wordspace    1911
Name: count, dtype: int64

Could be usefull for another project but these features aren't really what I'm interested in here.

In [10]:
tags_df[tags_df["tag_category"] == "Theme"]["tag_value"].value_counts()

tag_value
Wacky          119
Techno         102
Brush           76
Distressed      61
Woodtype        40
Pixel           35
Blobby          33
Blackletter     31
Inline          19
Medieval        18
Stencil         17
Art Deco        16
Shaded          13
Art Nouveau      7
Tuscan           6
Name: count, dtype: int64

In [11]:
tags_df[tags_df["tag_category"] == "Expressive"]["tag_value"].value_counts()

tag_value
Loud             1805
Business         1006
Happy             807
Vintage           640
Stiff             463
Calm              461
Active            417
Futuristic        412
Sincere           353
Playful           319
Competent         300
Cute              280
Artistic          240
Excited           189
Awkward           188
Innovative        156
Childlike         146
Rugged            113
Fancy              97
Sophisticated      66
Name: count, dtype: int64

From the numbers here the expressive category seems to be the most filled out. This is good since it was what I was planning on using.

### Filtering down to expressive

In [12]:
tags_df_filtered = tags_df[tags_df["tag_category"] == "Expressive"]

In [13]:
print(tags_df_filtered.head())

         font                   tag  value tag_category tag_value
0  42dot Sans  /Expressive/Business   75.0   Expressive  Business
1  42dot Sans      /Expressive/Loud   40.0   Expressive      Loud
6     ABeeZee    /Expressive/Active   10.0   Expressive    Active
7     ABeeZee  /Expressive/Business   75.0   Expressive  Business
8     ABeeZee      /Expressive/Calm   91.6   Expressive      Calm


In [32]:
tags_df_filtered[["font", "tag_value", "value"]].pivot_table(index="font", columns="tag_value", values="value").fillna(0).to_csv("tags/expressive_tags.csv")

In [14]:
# Pivot using 'font' as index, 'tag' (or 'tag_value') as columns, 'value' as values
pivoted = tags_df_filtered.pivot_table(index='font', columns='tag_value', values='value')

print(pivoted)

tag_value                  Active  Artistic  Awkward  Business  Calm  \
font                                                                   
42dot Sans                    NaN       NaN      NaN      75.0   NaN   
ABeeZee                      10.0       NaN      NaN      75.0  91.6   
ADLaM Display                10.0       NaN      NaN       NaN   NaN   
AR One Sans                   5.0       NaN      NaN      75.0  90.6   
Abel                         10.0       NaN      NaN       NaN  83.4   
...                           ...       ...      ...       ...   ...   
Zeyada                       20.0      71.0     55.0       NaN   NaN   
Zhi Mang Xing                67.0      67.0      NaN       NaN   NaN   
Zilla Slab                    NaN       NaN      NaN      88.5   NaN   
Zilla Slab Highlight          NaN       NaN      NaN       NaN   NaN   
Znamenny Musical Notation     NaN       NaN      NaN      80.0  99.0   

tag_value                  Childlike  Competent  Cute  Excited 

In [15]:
tag_corr = pivoted.corr()

In [16]:
tag_corr

tag_value,Active,Artistic,Awkward,Business,Calm,Childlike,Competent,Cute,Excited,Fancy,Futuristic,Happy,Innovative,Loud,Playful,Rugged,Sincere,Sophisticated,Stiff,Vintage
tag_value,,,,,,,,,,,,,,,,,,,,
Active,1.000000,0.076208,-0.239548,-0.222335,-0.093766,-0.263389,-0.225105,-0.189656,0.112785,-0.077105,-0.173711,-0.059920,0.146519,0.081436,-0.055797,0.027330,0.043829,-0.042281,0.193599,-0.168027
Artistic,0.076208,1.000000,0.114492,0.326459,NaN,0.058502,NaN,-0.027233,0.145982,0.295291,0.607950,0.047020,0.444582,-0.008528,0.379982,0.530238,-0.206744,0.083257,-0.012385,0.076093
Awkward,-0.239548,0.114492,1.000000,-0.526537,NaN,-0.302981,-0.520401,-0.080622,0.379814,NaN,0.192495,0.025964,0.020630,0.058882,0.546577,0.235952,0.040561,NaN,-0.286336,-0.118193
Business,-0.222335,0.326459,-0.526537,1.000000,0.286412,0.435501,0.013080,0.278909,0.307169,NaN,-0.384098,-0.183173,-0.470446,-0.136942,0.039877,-0.257757,0.430705,NaN,-0.110649,-0.062496
Calm,-0.093766,NaN,NaN,0.286412,1.000000,-1.000000,0.086658,-0.163167,NaN,NaN,-0.055622,-0.423589,NaN,0.126341,0.272162,1.000000,-0.333935,NaN,-0.060290,0.039603
Childlike,-0.263389,0.058502,-0.302981,0.435501,-1.000000,1.000000,NaN,-0.032414,-0.141994,NaN,-0.221311,-0.087729,-0.194163,-0.184513,-0.003070,0.302668,0.487307,NaN,-0.126444,-0.060395
Competent,-0.225105,NaN,-0.520401,0.013080,0.086658,NaN,1.000000,0.367366,1.000000,NaN,0.154735,-0.180968,1.000000,0.068864,0.036951,NaN,0.184763,NaN,0.075617,0.295624
Cute,-0.189656,-0.027233,-0.080622,0.278909,-0.163167,-0.032414,0.367366,1.000000,0.163176,-0.083419,0.086086,0.233924,-0.034398,-0.046411,0.089587,-0.118243,0.031748,0.223279,-0.548780,0.043648
Excited,0.112785,0.145982,0.379814,0.307169,NaN,-0.141994,1.000000,0.163176,1.000000,-0.051131,-0.097739,0.093108,0.456133,-0.098542,0.193331,0.114785,0.113522,-0.913682,0.216327,0.025743


A lot of the columns with the highest negative correlations aren't what you'd expect normally expect.

In [22]:
print("Negatively correlated values can be opposites on a scale. Do we want 0-100 or -100 to +100?")

upper = tag_corr.where(np.triu(np.ones(tag_corr.shape), k=1).astype(bool))

print("Strongly correlated negatively:\n")

pairs = []
for col in upper.columns:
    for row in upper.index:
        if upper.loc[col, row] < -0.3:
            pairs.append((col, row, upper.loc[col, row]))

for col, row, val in sorted(pairs, key=lambda x: x[2]):
    print(col, row, val)

Negatively correlated values can be opposites on a scale. Do we want 0-100 or -100 to +100?
Strongly correlated negatively:

Calm Childlike -1.0
Excited Sophisticated -0.913681575507151
Innovative Sincere -0.6817943093803575
Cute Stiff -0.5487803134891395
Awkward Business -0.5265373066917453
Awkward Competent -0.5204014838815378
Playful Sophisticated -0.49705108830399736
Business Innovative -0.4704461764509115
Futuristic Sincere -0.4693119940907421
Calm Happy -0.423589307577666
Business Futuristic -0.3840980264903996
Calm Sincere -0.33393486265861233
Futuristic Rugged -0.32104821071909334
Awkward Childlike -0.3029814185277465


In [25]:
print("Negatively correlated values can be opposites on a scale. Do we want 0-100 or -100 to +100?")

upper = tag_corr.where(np.triu(np.ones(tag_corr.shape), k=1).astype(bool))

print("Strongly correlated positively:\n")

pairs = []
for col in upper.columns:
    for row in upper.index:
        if upper.loc[col, row] > 0.3:
            pairs.append((col, row, upper.loc[col, row]))

for col, row, val in sorted(pairs, key=lambda x: x[2], reverse=True):
    print(col, row, val)

Negatively correlated values can be opposites on a scale. Do we want 0-100 or -100 to +100?
Strongly correlated positively:

Calm Rugged 1.0
Competent Excited 1.0
Competent Innovative 0.999999999999997
Artistic Futuristic 0.6079496955931125
Playful Rugged 0.5481365971196761
Awkward Playful 0.5465773963789419
Artistic Rugged 0.5302378159852432
Futuristic Playful 0.5190313245387457
Childlike Sincere 0.48730673927084744
Futuristic Stiff 0.47435364894220583
Excited Innovative 0.45613262761629864
Artistic Innovative 0.44458171596426455
Business Childlike 0.43550124021590475
Business Sincere 0.43070526589402724
Fancy Sophisticated 0.40356200688287736
Innovative Playful 0.3973654331444362
Artistic Playful 0.3799823806898772
Awkward Excited 0.37981356290157314
Competent Cute 0.36736648818304923
Innovative Rugged 0.3658690511462782
Happy Playful 0.3374884233927166
Sophisticated Vintage 0.33534614890991515
Artistic Business 0.3264594007425237
Business Excited 0.30716944988734557
Childlike Rugged

### What all this means
Correlation is not perfect due to the small sample size (~1900 fonts)

Some of these tags appear a lot less frequently than others: `sophisticated` appears only 66 times where as `buisness` appears 1006 times.

This difference in number of appearances makes strange correlations appear where they shouldn't for example Calm & Rugged (1) or Buisness & Childlike (0.43)

TLDR: Correlation can't be used to create sliders because sample size is too small. Need to make a semantic slider myself. 

The results from testing zeroshot training with CLIP are bad.

# FINAL SLIDER LIST

Formality        
Energy           
Age              Old Timey <-> Futuristic

In [41]:
tags_df[tags_df["tag_category"] == "Expressive"]["tag_value"].value_counts().sort_index()

tag_value
Active            417
Artistic          240
Awkward           188
Business         1006
Calm              461
Childlike         146
Competent         300
Cute              280
Excited           189
Fancy              97
Futuristic        412
Happy             807
Innovative        156
Loud             1805
Playful           319
Rugged            113
Sincere           353
Sophisticated      66
Stiff             463
Vintage           640
Name: count, dtype: int64